# Sensitivity Analysis (Input Language) Justice Principles: Multilingual Descriptive Analysis


This notebook extends the Main Experiment descriptive workflow to Sensitivity Analysis (Input Language) experiment runs. We replicate the full descriptive analysis for the three language cohorts (English, Spanish, Mandarin), retaining the a colorblind-accessible visual system and comparing outputs across groups.


In [ ]:
import sys
import json
from pathlib import Path
from typing import Any, Dict, List, Tuple
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

_NOTEBOOK_DIR = Path.cwd().resolve()
for candidate in [_NOTEBOOK_DIR, *_NOTEBOOK_DIR.parents]:
    if (candidate / "experiment_execution").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

# All imports from centralized package
from experiment_execution.utils_experiment_execution import (
    # Style
    COLORS,
    FIG_SIZES,
    FONT_SIZES,
    PRINCIPLE_COLORS,
    PRINCIPLE_DISPLAY_NAMES,
    apply_theme,
    format_principle_label,
    format_principle_labels,
    # Constants
    CERTAINTY_TO_SCORE,
    FINAL_WAVE,
    PRINCIPLE_LABELS,
    PRINCIPLE_ORDER,
    WAVE_DEFINITIONS,
    WAVE_ORDER,
    # Data loading
    load_experiment_runs,
    extract_run_metrics,
    extract_vote_rounds,
    extract_rankings,
    extract_income_classes,
    build_transition_data,
    create_transition_matrix,
    GroupDataset,
    build_group_dataset,
    # Visualizations
    plot_floor_constraint_distribution,
    plot_floor_constraint_distribution_grouped,
    plot_income_composition,
    plot_income_preference_bars,
    plot_long_term_counts_grid,
    plot_long_term_margin,
    plot_long_term_stability,
    plot_long_term_stability_grid,
    plot_preference_stability,
    plot_rounds_to_outcome,
    plot_rounds_to_outcome_grouped,
    plot_transition_heatmaps,
    plot_voting_attempts_summary,
)

apply_theme()


In [ ]:
LANGUAGE_GROUPS: "OrderedDict[str, str]" = OrderedDict([
    ("English", "english"),
    ("Spanish", "spanish"),
    ("Mandarin", "mandarin"),
])

# Use centralized constants (imported above)
# PRINCIPLE_LABELS, WAVE_ORDER, PRINCIPLE_ORDER are now available

PRINCIPLE_DISPLAY_ORDER = [PRINCIPLE_DISPLAY_NAMES[name] for name in PRINCIPLE_ORDER]


In [ ]:
def locate_project_root(markers: Tuple[str, ...] = ("experiment_execution", "config")) -> Path:
    """Locate the project root by walking up the directory tree."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError("Could not locate project root from current working directory.")


PROJECT_ROOT = locate_project_root()
RESULTS_ROOT = PROJECT_ROOT / "experiment_execution" / "sensitivity_analysis_input_language" / "results"
TERMINAL_ROOT = PROJECT_ROOT / "experiment_execution" / "sensitivity_analysis_input_language" / "terminal_outputs"


def load_language_group_runs(language_slug: str) -> List[Tuple[str, Dict[str, Any]]]:
    """Load runs for a specific language group."""
    return load_experiment_runs(
        results_dir=RESULTS_ROOT / language_slug,
        pattern=f"sensitivity_analysis_input_language_{language_slug}_condition_*_config_results.json",
        terminal_outputs_dir=TERMINAL_ROOT / language_slug,
        log_pattern=f"sensitivity_analysis_input_language_{language_slug}_condition_*_log",
    )


In [ ]:

# Load and preprocess each language cohort

group_datasets: "OrderedDict[str, GroupDataset]" = OrderedDict()

for label, slug in LANGUAGE_GROUPS.items():
    runs = load_group_runs(slug)
    dataset = build_group_dataset(label, slug, runs)
    transition_df, coverage = build_transition_data(dataset.ranking_top)
    dataset.transition_data = transition_df
    dataset.transition_meta = coverage

    income_df = extract_income_classes(runs, label)
    dataset.income_df = income_df
    dataset.switcher_analysis = prepare_switcher_analysis(transition_df, income_df)

    group_datasets[label] = dataset
    print(f"Loaded {len(runs)} runs for {label} (transition coverage: {coverage['complete_cases']}/{coverage['total_agents']} agents)")


In [ ]:
ordered_languages = ["English", "Mandarin", "Spanish"]
grouped_rounds_data = [
    (label, group_datasets[label].run_metrics if label in group_datasets else None)
    for label in ordered_languages
]

plot_rounds_to_outcome_grouped(
    grouped_rounds_data,
    language_order=ordered_languages,
    font_scale=1.3,
    annotation_fontsize=14,
)


In [ ]:

# Cohort-level overview table

overview_records: List[Dict[str, Any]] = []

for label, dataset in group_datasets.items():
    run_metrics = dataset.run_metrics
    consensus_runs = run_metrics[run_metrics["consensus_reached"] == True]
    rounds = consensus_runs["rounds_to_outcome"].dropna()
    overview_records.append({
        "Cohort": label,
        "Runs": run_metrics.shape[0],
        "Consensus Runs": consensus_runs.shape[0],
        "Consensus Rate": consensus_runs.shape[0] / run_metrics.shape[0] if run_metrics.shape[0] else np.nan,
        "Mean Rounds": rounds.mean() if not rounds.empty else np.nan,
        "Median Rounds": rounds.median() if not rounds.empty else np.nan,
    })

overview_df = pd.DataFrame(overview_records)
overview_df["Consensus Rate"] = (overview_df["Consensus Rate"] * 100).round(1)
overview_df["Mean Rounds"] = overview_df["Mean Rounds"].round(2)
overview_df["Median Rounds"] = overview_df["Median Rounds"].round(1)

display(Markdown("### High-Level Cohort Overview"))
display(overview_df)


In [ ]:

# Run full descriptive workflow per cohort

for label, dataset in group_datasets.items():
    display(Markdown(f"## {label} Cohort"))

    # Data summary
    display(Markdown("### Data Summary"))
    num_runs = dataset.run_metrics.shape[0]
    consensus_runs = dataset.run_metrics[dataset.run_metrics["consensus_reached"] == True].shape[0]
    unique_agents = dataset.ranking_top[["run_id", "agent"]].drop_duplicates().shape[0]
    num_waves = dataset.ranking_top["wave_label"].nunique()
    summary_df = pd.DataFrame(
        {
            "Metric": ["Runs", "Consensus runs", "Agent sessions", "Preference waves"],
            "Value": [num_runs, consensus_runs, unique_agents, num_waves],
        }
    )
    display(summary_df)
    if dataset.transition_meta:
        total = dataset.transition_meta.get("total_agents", 0)
        complete = dataset.transition_meta.get("complete_cases", 0)
        print(f"Transition coverage: {complete}/{total} agents with full wave data")

    # Preference orderings
    display(Markdown("### Preference Orderings by Wave"))
    for wave_name in WAVE_ORDER:
        display(Markdown(f"**{wave_name}**"))
        table = create_preference_ordering_table(dataset.ranking_long, wave_name)
        display(table)

    # Rounds to outcome
    display(Markdown("### Rounds to Outcome"))
    plot_rounds_to_outcome(
        dataset.run_metrics,
        title_suffix=label,
    )

    # Floor constraints
    display(Markdown("### Floor Constraint Amounts"))
    plot_floor_constraint_distribution(
        dataset.vote_rounds,
        title_suffix=label,
    )

    # Voting attempts
    display(Markdown("### Voting Attempts and Success Rate"))
    plot_voting_attempts_summary(
        dataset.run_metrics,
        title_suffix=label,
    )

    # Preference transitions
    display(Markdown("### Preference Evolution and Transitions"))
    plot_preference_stability(
        dataset.transition_data,
        title_suffix=label,
    )
    plot_transition_heatmaps(
        dataset.transition_data,
        principle_display_order=PRINCIPLE_DISPLAY_ORDER,
        format_principle_label=format_principle_label,
        title_suffix=label,
    )
    plot_long_term_stability(
        dataset.transition_data,
        principle_order=PRINCIPLE_ORDER,
        principle_display_order=PRINCIPLE_DISPLAY_ORDER,
        format_principle_label=format_principle_label,
        title_suffix=label,
        title=label,
    )
    plot_long_term_margin(
        dataset.transition_data,
        principle_display_order=PRINCIPLE_DISPLAY_ORDER,
        format_principle_label=format_principle_label,
        title_suffix=label,
    )

    # Switcher analysis
    display(Markdown("### Switcher Analysis: Income Class and Preference Changes"))
    if dataset.switcher_analysis is None or dataset.switcher_analysis.empty:
        print("No switcher data available for this cohort.")
    else:
        switcher_summary, count_long, percent_long, composition_percent_long = summarize_income_preferences(dataset.switcher_analysis)
        display(switcher_summary)
        plot_income_preference_bars(
            switcher_summary,
            count_long,
            percent_long,
            title_suffix=label,
        )
        plot_income_composition(
            count_long,
            composition_percent_long,
            title_suffix=label,
        )


In [ ]:

# Cross-cohort comparison of final top choices

final_choice_records: List[Dict[str, Any]] = []
for label, dataset in group_datasets.items():
    if dataset.transition_data is None or dataset.transition_data.empty:
        continue
    final_wave = dataset.transition_data["wave4"].map(format_principle_label)
    counts = final_wave.value_counts()
    total = counts.sum()
    for principle, count in counts.items():
        final_choice_records.append({
            "Cohort": label,
            "Top Principle": principle,
            "Agents": int(count),
            "Share %": round(count / total * 100, 1) if total else np.nan,
        })

final_choice_df = pd.DataFrame(final_choice_records).sort_values(["Top Principle", "Cohort"])

display(Markdown("### Final Wave Top-Choice Distribution"))
display(final_choice_df)


## Direct Comparison

In [ ]:
comparison_data = [
    (label, dataset.transition_data)
    for label, dataset in group_datasets.items()
]

plot_long_term_stability_grid(
    comparison_data,
    principle_order=PRINCIPLE_ORDER,
    principle_display_order=PRINCIPLE_DISPLAY_ORDER,
    format_principle_label=format_principle_label,
    title="Long-Term Preference Stability (Wave 1 → Wave 4)",
)


### Counts-Only Comparison

In [ ]:
ordered_languages = ["English", "Mandarin", "Spanish"]
counts_only_data = [
    (label, group_datasets[label].transition_data if label in group_datasets else None)
    for label in ordered_languages
]

plot_long_term_counts_grid(
    counts_only_data,
    principle_display_order=PRINCIPLE_DISPLAY_ORDER,
    format_principle_label=format_principle_label,
    orientation="horizontal",
    axis_title_fontsize=16,

    colorbar_mode="shared",
)




In [ ]:
grouped_rounds_data = [
    (label, group_datasets[label].run_metrics if label in group_datasets else None)
    for label in ordered_languages
]

plot_rounds_to_outcome_grouped(
    grouped_rounds_data,
    language_order=ordered_languages,
    font_scale=1.3
)


In [ ]:
grouped_constraint_data = [
    (label, group_datasets[label].vote_rounds if label in group_datasets else None)
    for label in ordered_languages
]

plot_floor_constraint_distribution_grouped(
    grouped_constraint_data,
    language_order=ordered_languages,
    target_principle_label="Max Avg + Floor",
    font_scale=1.3,
    bin_width=4000,
    show_title=False,
    annotation_fontsize=14,
)


In [ ]:
grouped_constraint_data = [
    (label, group_datasets[label].vote_rounds if label in group_datasets else None)
    for label in ordered_languages
]

plot_floor_constraint_distribution_grouped(
    grouped_constraint_data,
    language_order=ordered_languages,
    target_principle_label="Max Avg + Floor",
    font_scale=1.3,
    bin_width=4000,
    show_title=False,
)